# Revision: VQ-VAE day-type occupancy by clinical cohort group

Equal sample of **N users per clinical group** from `daily_summary_eb2prod_may26.csv`, encode with the same pipeline as `daily_to_profiles.ipynb`, one profile pickle per group:

```text
profiles[mode][top_n][user_id] -> [(length, seq, mapped, dates, some_obs, orig_info, top_info), ...]
```

Groups: **CI, das, DM, ED1, ED2, GM, PMP, SR, TMC**. **CNIO** is kept aparte (not in the equal sample). Groups with &lt; N users (e.g. DM) are skipped automatically.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "scripts").exists():
    ROOT = ROOT.parent
if not (ROOT / "scripts").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from scripts.vqvae import constants as c
from scripts.vqvae.sample_cohort_profiles import (
    COHORT_GROUPS,
    DEFAULT_CSV_DIR,
    DEFAULT_OUT_DIR,
    DEFAULT_SOURCE,
    cohort_group_counts,
    encode_cohort_csvs,
    sample_users_by_cohort_group,
    save_manifest,
    write_model_input_csvs,
)

N_PER_COHORT = 90
SEED = c.DEFAULT_SEED
# Optional subset of codes, e.g. ["CI", "das", "ED1", "PMP"]; None = all sampleable groups
COHORTS = None
SAMPLE_ONLY = False
DEVICE = "cpu"
MODES = ("a0",)

SOURCE_CSV = DEFAULT_SOURCE
CSV_DIR = DEFAULT_CSV_DIR
OUT_DIR = DEFAULT_OUT_DIR

assert SOURCE_CSV.exists(), f"Missing source CSV: {SOURCE_CSV}"
print("source:", SOURCE_CSV)
print("groups:", list(COHORT_GROUPS))
print("csv_dir:", CSV_DIR)
print("out_dir:", OUT_DIR)

## 1. Clinical cohort group sizes

Equal sample uses groups with `sample=True` and ≥ `N_PER_COHORT` users. CNIO is aparte.

In [ ]:
counts = cohort_group_counts(SOURCE_CSV)
display(counts)
eligible = counts[(counts["sample"]) & (counts["n_users"] >= N_PER_COHORT)]
print(f"Equal-sample eligible (>= {N_PER_COHORT}): {', '.join(eligible['cohort'])}")
print("Aparte / skipped:", ", ".join(counts.loc[~counts["sample"], "cohort"]))
print(
    "Undersized:",
    ", ".join(
        counts.loc[(counts["sample"]) & (counts["n_users"] < N_PER_COHORT), "cohort"]
    )
    or "(none)",
)

## 2. Equal sample per group + model-input CSVs

Same feature columns as oncology VQ-VAE input (`scripts.vqvae.constants.COLS`).

In [ ]:
selected = sample_users_by_cohort_group(
    SOURCE_CSV,
    n_per_cohort=N_PER_COHORT,
    cohorts=COHORTS,
    seed=SEED,
)
manifest_path = save_manifest(
    selected, OUT_DIR, N_PER_COHORT, SEED, SOURCE_CSV, group_counts=counts
)
print("sampled:", list(selected))
print("manifest:", manifest_path)

csv_paths = write_model_input_csvs(SOURCE_CSV, selected, CSV_DIR, clean=True)
pd.DataFrame(
    [
        {
            "cohort": name,
            "label": COHORT_GROUPS[name]["name"],
            "n_users": pd.read_csv(path, usecols=["user"])["user"].nunique(),
            "csv": str(path.relative_to(ROOT)),
        }
        for name, path in csv_paths.items()
    ]
)

## 3. Encode with VQ-VAE (`generate_profiles`)

Same checkpoints + scaler as `daily_to_profiles.ipynb`. Writes `profiles_per_sample_<cohort>.pkl` under `data/output_vq_vae/revision_cohorts/`.

In [ ]:
if SAMPLE_ONLY:
    print("SAMPLE_ONLY=True — skip encoding.")
    pkl_paths = {}
else:
    pkl_paths = encode_cohort_csvs(
        csv_paths,
        OUT_DIR,
        modes=MODES,
        device=DEVICE,
        seed=SEED,
        clean_pickles=True,
    )
    display(
        pd.DataFrame(
            [
                {
                    "cohort": k,
                    "label": COHORT_GROUPS[k]["name"],
                    "pkl": str(v.relative_to(ROOT)),
                }
                for k, v in pkl_paths.items()
            ]
        )
    )

## 4. Quick occupancy check (shared codebook usage)

For each cohort, count how often each day-type (embedding id) appears under mode `a0`, window `30`.

## 5. Comparison: occupancy + decode

**How we compare (recommended pair of views):**

1. **Occupancy** — probability of each of the 256 VQ-VAE day-types within a cohort; heatmaps of most-used and enriched codes; JS distance between cohorts.
2. **Decode** — map day-types through `decoded_embedding_vectors_a0.pkl` to behavioral features; usage-weighted mean signature per cohort, and decoded top day-types.

**CNIO:** `cnio_caspar` in the train file has only ~11 users, so **CNIO = 90 patients sampled from** `oncology_daily_summary_model_input.csv` (study oncology cohort).

In [ ]:
from IPython.display import Image, display
from scripts.vqvae.compare_cohort_profiles import (
    DEFAULT_FIG_DIR,
    DEFAULT_OUT_DIR,
    run_comparison,
)

comp_paths = run_comparison(out_dir=DEFAULT_OUT_DIR, fig_dir=DEFAULT_FIG_DIR, top_k=15)

occ = pd.read_csv(comp_paths["occupancy_csv"], index_col=0)
weighted = pd.read_csv(comp_paths["weighted_decoded_csv"])
js = pd.read_csv(comp_paths["js_csv"], index_col=0)
top = pd.read_csv(comp_paths["top_profiles_csv"])

print("Artifacts:")
for k, p in comp_paths.items():
    print(f"  {k}: {p.relative_to(ROOT)}")

display(weighted)
display(js.round(3))
display(top.groupby("cohort").head(5)[["cohort", "rank", "day_type", "occupancy"]])

for key in ["fig_occupancy", "fig_enrichment", "fig_decoded", "fig_top_decoded", "fig_js"]:
    display(Image(filename=str(comp_paths[key])))

In [ ]:
import pickle
from collections import Counter

MODE = "a0"
TOP_N = 30

def embedding_counts_from_pkl(path: Path, mode=MODE, top_n=TOP_N) -> Counter:
    with path.open("rb") as f:
        profiles = pickle.load(f)
    counts = Counter()
    for user_id, entries in profiles[mode][top_n].items():
        length, seq, *_ = entries[0]
        counts.update(int(x) for x in seq[:length])
    return counts

if pkl_paths:
    summary = []
    for cohort, path in pkl_paths.items():
        cnt = embedding_counts_from_pkl(path)
        summary.append(
            {
                "cohort": cohort,
                "n_day_tokens": sum(cnt.values()),
                "n_unique_day_types": len(cnt),
                "top5_day_types": cnt.most_common(5),
            }
        )
    display(pd.DataFrame(summary))
else:
    print("No pickles yet — set SAMPLE_ONLY=False and re-run encoding.")